### This notebook is for the one-time loading of USGS basin polygon locations into the teehr data warehouse
- Use location ID prefix: `usgsbasin-<usgs gage id>`
- NWM 3.0 retrospective RAINRATE (AORC but maybe slightly different?) will be saved as `primary_timeseries`
- No crosswalk data was added at this time

In [1]:
from pathlib import Path

import geopandas as gpd
import pandas as pd

import teehr
from teehr.evaluation.spark_session_utils import create_spark_session

In [2]:
BASINS_FILEPATH = Path("/data/common/geometry/usgs_basins/usgsbasin_geometry_04202026.parquet")

In [3]:
%%time
spark = create_spark_session(
    aws_profile="default",
)
ev = teehr.RemoteReadWriteEvaluation(spark=spark)

INFO:teehr.evaluation.spark_session_utils:🚀 Creating Spark session: TEEHR Evaluation
INFO:teehr.evaluation.spark_session_utils:✅ Spark local configuration successful!
INFO:teehr.evaluation.spark_session_utils:Setting Hadoop's default AWS credentials provider and AWS region
INFO:teehr.evaluation.spark_session_utils:🔑 Using AWS credentials from ~/.aws/credentials profile 'admin-user
INFO:teehr.evaluation.spark_session_utils:Configuring Iceberg catalogs...
INFO:teehr.evaluation.spark_session_utils:⚙️ All settings applied. Creating Spark session...
INFO:teehr.evaluation.spark_session_utils:🎉 Spark session created successfully!
INFO:teehr.evaluation.evaluation:Using provided Spark session.
INFO:teehr.evaluation.evaluation:Active catalog set to iceberg.


CPU times: user 110 ms, sys: 68.6 ms, total: 178 ms
Wall time: 19.1 s


#### Locations

In [4]:
gdf = gpd.read_parquet(BASINS_FILEPATH)  # NOTE: We should include EPSG/CRS in the locations table? (table properties or an extra field?)
gdf.crs

<Geographic 2D CRS: EPSG:4326>
Name: WGS 84
Axis Info [ellipsoidal]:
- Lat[north]: Geodetic latitude (degree)
- Lon[east]: Geodetic longitude (degree)
Area of Use:
- name: World.
- bounds: (-180.0, -90.0, 180.0, 90.0)
Datum: World Geodetic System 1984 ensemble
- Ellipsoid: WGS 84
- Prime Meridian: Greenwich

In [5]:
gdf.head()

,id,name,geometry,properties
0,usgsbasin-01010000,"St. John River at Ninemile Bridge, Maine","POLYGON ((-70.21546 46.10614, -70.21796 46.104...","{'date_calculated': None, 'date_retrieved': '0..."
1,usgsbasin-01010070,"Big Black River near Depot Mtn, Maine","POLYGON ((-69.76602 46.87059, -69.76656 46.869...","{'date_calculated': None, 'date_retrieved': '0..."
2,usgsbasin-01010500,"St. John River at Dickey, Maine","POLYGON ((-69.98691 46.03205, -69.98713 46.034...","{'date_calculated': None, 'date_retrieved': '0..."
3,usgsbasin-01011000,"Allagash River near Allagash, Maine","POLYGON ((-69.14836 46.31136, -69.14489 46.310...","{'date_calculated': None, 'date_retrieved': '0..."
4,usgsbasin-01013500,"Fish River near Fort Kent, Maine","POLYGON ((-68.71709 47.17016, -68.71906 47.172...","{'date_calculated': None, 'date_retrieved': '0..."


In [6]:
gdf.index.size

10588

In [7]:
gdf["geometry"] = gdf.geometry.make_valid()

Append to the locations table in the warehouse

In [8]:
%%time
ev.locations.load_dataframe(
    df=gdf,
    write_mode="append"
)

INFO:teehr.evaluation.tables.base_table:Initializing Table for table: locations.
INFO:teehr.evaluation.tables.base_table:Loading files from iceberg.teehr.locations.
INFO:teehr.evaluation.read:Reading files from iceberg.teehr.locations.
INFO:teehr.evaluation.tables.generic_table:Getting table: locations.
INFO:teehr.evaluation.tables.base_table:Initializing Table for table: locations.
INFO:teehr.evaluation.tables.base_table:Loading files from iceberg.teehr.locations.
INFO:teehr.evaluation.read:Reading files from iceberg.teehr.locations.
INFO:teehr.evaluation.validate:Start enforcing dataframe schema.
INFO:teehr.evaluation.validate:Validating DataFrame against schema.
INFO:teehr.evaluation.validate:Finished enforcing dataframe schema in 6.452 seconds.
INFO:teehr.evaluation.write:Start writing to warehouse table 'locations'.
INFO:teehr.evaluation.tables.generic_table:Getting table: locations.
INFO:teehr.evaluation.tables.base_table:Initializing Table for table: locations.
INFO:teehr.evalua

CPU times: user 1.22 s, sys: 66.5 ms, total: 1.28 s
Wall time: 13.3 s


In [11]:
gdf["properties"].iloc[0]

{'date_calculated': None,
 'date_retrieved': '04-03-2026',
 'method': None,
 'simplify_tolerance': 'tol: 0.0002, (1401 reduced to 1065 coords)',
 'source': 'HydroShare',
 'url': 'https://www.hydroshare.org/resource/b26b7c34c2e94d9087119ae1506b3e11/'}